# Ejercicio calidad de datos

# Ejercicio: Evaluación de Calidad de Datos - Dataset de Vuelos

## Objetivo
Evaluar la calidad del dataset `flights.parquet` identificando problemas comunes de calidad de datos y proponiendo soluciones.

## Contexto
El dataset contiene información sobre vuelos realizados en 2014, incluyendo horarios, aerolíneas, aeropuertos y retrasos.

## Tareas a Realizar

1. Exploración Inicial
2. **Eliminación de Columnas No Significativas**
3. **Tratamiento de Valores Faltantes**
4. **Detección y Tratamiento de Outliers**
5. **Corrección de Tipos de Datos**
6. **Eliminación de Duplicados**
7. **Validación de Consistencia**
8. **Enriquecimiento de Datos**
9. **Exportación y Persistencia**

## Solución

In [ ]:
import pandas as pd

# Cargar el dataset
df = pd.read_parquet('../data/flights.parquet')

In [ ]:
# 1. Exploración Inicial
print("=" * 50)
print("1. EXPLORACIÓN INICIAL")
print("=" * 50)
print(f"\nDimensiones del dataset: {df.shape}")
print(f"\nPrimeras filas:\n{df.head()}")
print(f"\nInformación del dataset:")
print(df.info())
print(f"\nEstadísticas descriptivas:\n{df.describe()}")
print(f"\nValores faltantes por columna:\n{df.isnull().sum()}")

In [ ]:
# 2. Eliminación de Columnas No Significativas
print("\n" + "=" * 50)
print("2. ELIMINACIÓN DE COLUMNAS NO SIGNIFICATIVAS")
print("=" * 50)
# Identificar columnas con todos valores nulos o con un único valor
cols_to_drop = []
for col in df.columns:
    if df[col].isnull().all() or df[col].nunique() == 1:
        cols_to_drop.append(col)
print(f"\nColumnas a eliminar: {cols_to_drop}")
df = df.drop(columns=cols_to_drop)
print(f"Nuevas dimensiones: {df.shape}")

In [ ]:
# 3. Tratamiento de Valores Faltantes
print("\n" + "=" * 50)
print("3. TRATAMIENTO DE VALORES FALTANTES")
print("=" * 50)
print(f"\nValores faltantes antes del tratamiento:\n{df.isnull().sum()}")
# Imputar valores numéricos con la mediana
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
# Imputar valores categóricos con la moda
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)
print(f"\nValores faltantes después del tratamiento:\n{df.isnull().sum()}")


In [ ]:

# 4. Detección y Tratamiento de Outliers
print("\n" + "=" * 50)
print("4. DETECCIÓN Y TRATAMIENTO DE OUTLIERS")
print("=" * 50)
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
    if len(outliers) > 0:
        print(f"\nColumna '{col}': {len(outliers)} outliers detectados")
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)


In [ ]:
# 5. Corrección de Tipos de Datos
print("\n" + "=" * 50)
print("5. CORRECCIÓN DE TIPOS DE DATOS")
print("=" * 50)
print(f"\nTipos de datos actuales:\n{df.dtypes}")
# Convertir columnas de fecha si existen
date_columns = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
for col in date_columns:
    try:
        df[col] = pd.to_datetime(df[col])
        print(f"Columna '{col}' convertida a datetime")
    except:
        pass


In [ ]:
# 6. Eliminación de Duplicados
print("\n" + "=" * 50)
print("6. ELIMINACIÓN DE DUPLICADOS")
print("=" * 50)
duplicates_before = df.duplicated().sum()
print(f"\nFilas duplicadas encontradas: {duplicates_before}")
df = df.drop_duplicates()
print(f"Dimensiones después de eliminar duplicados: {df.shape}")


In [ ]:

# 7. Validación de Consistencia
print("\n" + "=" * 50)
print("7. VALIDACIÓN DE CONSISTENCIA")
print("=" * 50)
print(f"\nResumen de calidad final:")
print(f"Total de filas: {len(df)}")
print(f"Total de columnas: {len(df.columns)}")
print(f"Valores nulos totales: {df.isnull().sum().sum()}")


In [ ]:

# 8. Enriquecimiento de Datos
print("\n" + "=" * 50)
print("8. ENRIQUECIMIENTO DE DATOS")
print("=" * 50)
# Crear nuevas columnas si es posible
if 'dep_delay' in df.columns and 'arr_delay' in df.columns:
    df['total_delay'] = df['dep_delay'] + df['arr_delay']
    print("\nColumna 'total_delay' creada")


In [ ]:
# 9. Exportación y Persistencia
print("\n" + "=" * 50)
print("9. EXPORTACIÓN Y PERSISTENCIA")
print("=" * 50)
df.to_csv('flights14_clean.csv', index=False)
print("\nDataset limpio exportado como 'flights14_clean.csv'")
print(f"\nProceso completado. Dataset final: {df.shape}")